# Lab 01 — SUMO Foundation + TraCI + Official Bologna/Acosta

This notebook runs only after Lab 00. It uses a deterministic unit intersection for controller validation and the official Bologna/Acosta scenario for the canonical-paper real-world track.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y sumo sumo-tools git
!pip -q install "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "pyyaml>=6,<7" "pytest>=8,<9"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json, shutil

CANDIDATES=[Path('/content/DhakaFlow-Nexus-Labs-00-01'), Path('/content/drive/MyDrive/DhakaFlow-Nexus-Labs-00-01')]
REPO_ROOT=next((p for p in CANDIDATES if p.exists()),None)
if REPO_ROOT is None: raise FileNotFoundError('Upload/unzip project to /content or MyDrive first')
LAB=REPO_ROOT/'paper_labs/01_sumo_foundation'
DRIVE=Path('/content/drive/MyDrive/DhakaFlow-Nexus-Data/Lab01_SUMO')
OUT=DRIVE/'outputs'; OUT.mkdir(parents=True,exist_ok=True)
os.environ['SUMO_HOME']='/usr/share/sumo'
print('REPO_ROOT=',REPO_ROOT)
print('DRIVE=',DRIVE)

## 1. Environment and canonical paper

In [ ]:
print(subprocess.check_output(['sumo','--version'],text=True).splitlines()[0])
print(subprocess.check_output(['netconvert','--version'],text=True).splitlines()[0])
subprocess.run([sys.executable,str(LAB/'paper/fetch_open_paper.py')],check=False)

## 2. Static tests before simulation

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',str(LAB/'tests')],check=True,env={**os.environ,'PYTHONPATH':str(LAB)})

## 3. Build and validate deterministic unit intersection

In [ ]:
SC=LAB/'scenarios/unit_intersection'
subprocess.run(['bash','build.sh'],cwd=SC,check=True)
print('network exists:',(SC/'network.net.xml').exists())

## 4. Same-demand controller benchmark: Fixed-Time vs Actuated vs Max-Pressure

In [ ]:
UNIT_OUT=OUT/'unit_benchmark'
cmd=[sys.executable,'-m','src.run_unit_benchmark','--config',str(SC/'unit.sumocfg'),'--out',str(UNIT_OUT),'--seed','2018']
subprocess.run(cmd,cwd=LAB,check=True,env={**os.environ,'PYTHONPATH':str(LAB),'SUMO_HOME':'/usr/share/sumo'})
import pandas as pd
display(pd.read_csv(UNIT_OUT/'benchmark.csv'))

## 5. Fetch official SUMO Bologna/Acosta scenario
This is the real-world scenario track from the official SUMO scenarios repository. Upstream files remain untouched.

In [ ]:
OFFICIAL=DRIVE/'official'
subprocess.run([sys.executable,'-m','src.fetch_official_bologna','--dest',str(OFFICIAL)],cwd=LAB,check=True,env={**os.environ,'PYTHONPATH':str(LAB)})
REPO=OFFICIAL/'sumo-scenarios'
SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
BOLOGNA=REPO/'bologna/acosta'
print('upstream SHA=',SHA)
print('scenario=',BOLOGNA)

## 6. Inspect official scenario configs before running

In [ ]:
configs=sorted(BOLOGNA.rglob('*.sumocfg'))
print('configs found:')
for p in configs: print(' -',p.relative_to(BOLOGNA))
if not configs: raise RuntimeError('No official Bologna .sumocfg found; inspect upstream repo layout.')

## 7. Run official Bologna baseline unchanged

In [ ]:
BOL_OUT=OUT/'official_bologna'
cmd=[sys.executable,'-m','src.run_official_bologna','--scenario',str(BOLOGNA),'--out',str(BOL_OUT)]
subprocess.run(cmd,cwd=LAB,check=True,env={**os.environ,'PYTHONPATH':str(LAB),'SUMO_HOME':'/usr/share/sumo'})
print((BOL_OUT/'metrics.json').read_text())

## 8. Reproducibility manifest

In [ ]:
manifest={
 'sumo_version':subprocess.check_output(['sumo','--version'],text=True).splitlines()[0],
 'netconvert_version':subprocess.check_output(['netconvert','--version'],text=True).splitlines()[0],
 'official_sumo_scenarios_commit':SHA,
 'official_bologna_path':str(BOLOGNA),
 'unit_benchmark':str(UNIT_OUT/'benchmark.csv'),
 'official_bologna_metrics':str(BOL_OUT/'metrics.json'),
}
(OUT/'lab01_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))

## 9. Exit gate
Claude must inspect SUMO logs, unit phase transitions, route validity, benchmark metrics, and official Bologna execution. Lab status is not PASS merely because cells completed.